# 어간 추출과 표제어 추출: 영어와 한국어 전처리 비교

## 어간 추출과 표제어 추출은 왜 필요한가

같은 단어도 문법에 따라 `run`, `runs`, `running`, `ran`처럼 형태가 달라진다. 컴퓨터는 이를 서로 다른 단어로 세기 때문에 어휘 수가 커지고 빈도가 여러 항목으로 흩어진다. 어간 추출과 표제어 추출은 이러한 활용형을 대표 형태로 모아 검색이나 BoW·TF-IDF 같은 빈도 기반 분석에 활용하는 **단어 정규화** 방법이다.

### 어간 추출(stemming)

접두사나 접미사를 규칙으로 잘라 짧은 stem을 만든다. 처리 속도는 빠르지만 `swiftly → swiftli`처럼 실제 단어가 아닌 결과가 나오거나 불규칙 변화인 `ran → run`을 처리하지 못할 수 있다.

### 표제어 추출(lemmatization)

사전과 품사 정보를 이용해 `running`, `ran → run`, `was → be`처럼 사전의 기본형인 lemma를 찾는다. 결과를 해석하기 쉽지만 stemming보다 필요한 리소스와 계산량이 많다.

### 선택 기준

- 빠른 처리와 어휘 축소가 중요하면 stemming을 먼저 검토한다.
- 자연스러운 기본형과 해석이 중요하면 lemmatization을 먼저 검토한다.
- 감성이나 시제처럼 단어 형태가 중요한 과제에서는 정규화하지 않은 결과도 함께 비교한다.

한국어 형태소 분석기의 `stem`이나 `morphs()`는 영어의 stemming·lemmatization과 반환 형태가 다를 수 있으므로 각 API의 결과를 확인해야 한다.


## 실습 패키지와 영어 모델 설치

NLTK는 영어 stemming과 WordNet lemmatization에 사용하고, KoNLPy는 한국어 형태소 분석에 사용한다. spaCy 패키지와 `en_core_web_sm` 영어 모델은 서로 별도로 설치하므로 아래 셀에서 함께 준비한다.

KoNLPy의 `Mecab` 클래스는 MeCab Python 바인딩과 한국어 사전이 추가로 필요하다. `mecab-python3`는 Python과 MeCab 엔진을 연결하고, `mecab-ko-dic`은 한국어 형태소와 품사 정보를 제공한다.


In [26]:
%pip install -q nltk konlpy spacy mecab-python3 mecab-ko-dic

!python -m spacy download en_core_web_sm


Note: you may need to restart the kernel to use updated packages.
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 6.7 MB/s eta 0:00:02
     -------- ------------------------------- 2.6/12.8 MB 6.3 MB/s eta 0:00:02
     ---------------- ----------------------- 5.2/12.8 MB 8.6 MB/s eta 0:00:01
     --------------------------- ------------ 8.7/12.8 MB 10.5 MB/s eta 0:00:01
     ---------------------------------- ---- 11.3/12.8 MB 11.0 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 10.6 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### NLTK 데이터 리소스 다운로드

NLTK 패키지만 설치하면 토크나이저, 품사 태거와 WordNet 사전 데이터가 포함되지 않는다. `punkt`와 `punkt_tab`은 영어 토큰화, `averaged_perceptron_tagger_eng`는 영어 품사 태깅, `wordnet`은 품사에 따른 표제어 검색에 사용한다. 아래 셀은 각 리소스를 직접 다운로드하며 이미 준비된 리소스는 최신 상태라는 메시지만 출력한다.


In [27]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("wordnet")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\playdata2\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## 영어 어간 추출: Porter와 Lancaster 비교

`PorterStemmer.stem(token)`과 `LancasterStemmer.stem(token)`은 단어 하나를 받아 규칙으로 축약한 문자열 하나를 반환한다. Porter는 상대적으로 보수적이고 Lancaster는 더 공격적으로 자르는 경향이 있다. 같은 문장과 비교 단어를 두 알고리즘에 전달해 실제 단어가 아닌 stem, 불규칙 변화 처리 실패와 과도한 축약을 확인한다.

결과에서는 Porter의 `swiftly → swiftli`, Lancaster의 `were → wer`와 `organization → org`를 비교한다. 두 알고리즘 모두 사전을 사용하지 않으므로 불규칙 과거형 `ran`을 `run`으로 복원하지 못한다.


In [28]:
from nltk.stem import LancasterStemmer, PorterStemmer
from nltk.tokenize import word_tokenize

# 불규칙 과거형과 여러 접미사 형태를 한 문장에 넣어 성공 사례와 한계를 함께 확인한다.
sentence = (
    "The runners were running swiftly and easily. "
    "They ran past the finish line."
)

tokens = word_tokenize(sentence)

# 어간 추출 객체 생성
porter_stemmer = PorterStemmer()
lancaster_stemmer = LancasterStemmer()

porter_stems = [porter_stemmer.stem(token) for token in tokens]
lancaster_stems = [lancaster_stemmer.stem(token) for token in tokens]

print("원본 토큰: ", tokens)
print("Porter stem: ", porter_stems)
print("Lancaster stem: ", lancaster_stems)

# 활용형뿐 아니라 과도한 축약 차이가 잘 보이는 단어를 별도 표로 비교한다.
comparison_words = [
    "runners",
    "running",
    "swiftly",
    "easily",
    "organization",
    "maximum",
]
print("\n단어 / Porter / Lancaster")
for word in comparison_words:
    porter_stem = porter_stemmer.stem(word)
    lancaster_stem = lancaster_stemmer.stem(word)
    print(f"{word:>12} / {porter_stem:>10} / {lancaster_stem}")

원본 토큰:  ['The', 'runners', 'were', 'running', 'swiftly', 'and', 'easily', '.', 'They', 'ran', 'past', 'the', 'finish', 'line', '.']
Porter stem:  ['the', 'runner', 'were', 'run', 'swiftli', 'and', 'easili', '.', 'they', 'ran', 'past', 'the', 'finish', 'line', '.']
Lancaster stem:  ['the', 'run', 'wer', 'run', 'swift', 'and', 'easy', '.', 'they', 'ran', 'past', 'the', 'fin', 'lin', '.']

단어 / Porter / Lancaster
     runners /     runner / run
     running /        run / run
     swiftly /    swiftli / swift
      easily /     easili / easy
organization /      organ / org
     maximum /    maximum / maxim


## 품사를 지정하는 WordNet 표제어 추출

`WordNetLemmatizer.lemmatize(word, pos)`는 단어와 WordNet 품사를 받아 사전에 등록된 lemma를 반환한다. WordNet 품사는 명사 `'n'`, 동사 `'v'`, 형용사 `'a'`, 부사 `'r'`로 지정한다. 품사를 생략하면 기본값이 명사이므로 동사 `running`이나 `ran`이 원하는 기본형으로 바뀌지 않을 수 있다.

먼저 각 단어의 품사를 직접 지정해 `running`, `ran`, `runs`가 `run`으로, `am`, `is`, `were`가 `be`로 모이는지 확인한다. 형용사 비교에서는 `better → good`, `worse → bad`처럼 단순 접미사 제거로 처리하기 어려운 불규칙 변화를 살핀다.


In [29]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# 동사 단어 모음 -> 표제어 추출기에 전달 시 'v' 값을 같이 전달
verb_words = ["running", "ran", "runs", "have", "has", "am", "is", "are", "were"]

verb_lemmas = [lemmatizer.lemmatize(word, "v") for word in verb_words]
print("동사 원형: ", verb_words)
print("동사 lemma: ", verb_lemmas)

adjective_words = ["good", "better", "best", "bad", "worse", "worst"]

adjective_lemmas = [
    lemmatizer.lemmatize(word, "a") for word in adjective_words
]

print("형용사 원형: ", adjective_words)
print("형용사 lemma: ", adjective_lemmas)



동사 원형:  ['running', 'ran', 'runs', 'have', 'has', 'am', 'is', 'are', 'were']
동사 lemma:  ['run', 'run', 'run', 'have', 'have', 'be', 'be', 'be', 'be']
형용사 원형:  ['good', 'better', 'best', 'bad', 'worse', 'worst']
형용사 lemma:  ['good', 'good', 'best', 'bad', 'bad', 'bad']


### NLTK 품사 태그를 WordNet 품사로 변환하기

실제 문장에서는 각 단어의 품사를 손으로 지정하기 어렵다. `pos_tag(tokens)`는 단어별 Penn Treebank 품사를 반환하고, 이를 첫 글자 기준으로 WordNet의 `n`, `v`, `a`, `r`에 연결할 수 있다. 관사와 전치사처럼 대응 품사가 없는 토큰은 원래 표면형을 유지한다.

출력에서는 `runners/NNS → n`, `were/VBD → v`, `better/JJR → a`처럼 품사 체계가 변환되는 과정과 최종 lemma를 함께 확인한다. 이 변환이 품사 태거의 예측에 의존하므로 태그가 틀리면 lemma도 틀릴 수 있다.


In [30]:
# NLTK 품사를 WordNet 품사로 연결해 문장 단위 표제어 추출 수행

from nltk import pos_tag
from nltk.stem import WordNetLemmatizer  # Lemmatizer: 표제어 찾기 도구
from nltk.tokenize import word_tokenize

def nltk_pos_to_wordnet(nltk_pos):
    'Penn Treebank 태그를 WordNet의 n, v, a, r 또는 None으로 변환'
    if nltk_pos.startswith("N"):
        return "n"
    if nltk_pos.startswith("V"):
        return "v"
    if nltk_pos.startswith("J"):
        return "a"
    if nltk_pos.startswith("R"):
        return "r"
    # 대응 품사가 없는 관사·접속사·전치사·구두점은 None으로 표시한다.
    return None

text = "The runners were running happily and ran beside the others."
tokens = word_tokenize(text)
# print(tokens)

# 토큰마다 품사 찾기
tagged_tokens = pos_tag(tokens)
print(tagged_tokens)

lemmatizer = WordNetLemmatizer()

print("토큰 / NLTK POS / WordNet POS / Lemma")

sentence_lemmas = []

for token, nltk_pos in tagged_tokens:

    # nltk 품사 -> wordnet 일치하는 품사 찾기
    wordnet_pos = nltk_pos_to_wordnet(nltk_pos)

    # wordnet_pos가 있다면(품사가 있다면) 사전형을 찾고
    # 없으면 원래 토큰을 유지한다
    lemma = (
        lemmatizer.lemmatize(token.lower(), pos=wordnet_pos)
        if wordnet_pos  # None이 아니면 True, None이면 False
        else token.lower()
    )

    sentence_lemmas.append(lemma)
    # token:>10 == token을 출력한 공간을 10칸 확보하고 오른쪽 정렬(>) 해라
    print(f"{token:>10} / {nltk_pos:>3} / {wordnet_pos or '-':>3} / {lemma}")

print("문장 표제어: ", sentence_lemmas)

[('The', 'DT'), ('runners', 'NNS'), ('were', 'VBD'), ('running', 'VBG'), ('happily', 'RB'), ('and', 'CC'), ('ran', 'VBD'), ('beside', 'IN'), ('the', 'DT'), ('others', 'NNS'), ('.', '.')]
토큰 / NLTK POS / WordNet POS / Lemma
       The /  DT /   - / the
   runners / NNS /   n / runner
      were / VBD /   v / be
   running / VBG /   v / run
   happily /  RB /   r / happily
       and /  CC /   - / and
       ran / VBD /   v / run
    beside /  IN /   - / beside
       the /  DT /   - / the
    others / NNS /   n / others
         . /   . /   - / .
문장 표제어:  ['the', 'runner', 'be', 'run', 'happily', 'and', 'run', 'beside', 'the', 'others', '.']


## spaCy의 문맥 기반 표제어 추출

spaCy는 토큰화, 문맥 기반 품사 태깅과 lemmatization을 하나의 파이프라인으로 실행한다. `spacy.load("en_core_web_sm")`은 영어 분석 파이프라인을 불러오고, `nlp(text)`는 분석된 토큰을 담은 `Doc`을 반환한다. 각 토큰의 `pos_`는 보편 품사, `tag_`는 세분화된 영어 품사, `lemma_`는 모델이 선택한 표제어이다.

`WordNetLemmatizer`가 사용자가 품사를 전달하는 사전 기반 API라면 spaCy는 문장 전체를 보고 품사를 먼저 예측한다. 결과에서는 `running`과 `ran`이 모두 `run`으로 모이는지, 복수 명사가 단수 lemma로 바뀌는지 확인한다. 모델 버전에 따라 `better`처럼 사람이 기대한 결과와 다른 lemma가 나올 수 있으므로 POS와 lemma를 함께 읽는다.


In [31]:
import spacy

nlp = spacy.load("en_core_web_sm")

lemma_text = (
    "The runners were running and ran beside the cars. "
    "This result is better."
)

# 문맥 전체를 분석한 Doc 객체를 반환하고, Doc를 순회하면서 Token을 얻는다
lemma_doc = nlp(lemma_text)

focus_tokens = {"runners", "running", "ran", "cars", "better"}

# spaCy 출력 반복문의 실행 순서
# 1) lemma_doc에서 Token 객체를 원문 순서대로 하나씩 꺼낸다.
# 2) token.text.lower()가 focus_tokens 집합에 있는지 검사한다.
# 3) 조건을 통과한 Token만 text, pos_, tag_, lemma_ 속성을 한 줄로 출력한다.
for token in lemma_doc:
    if token.text.lower() in focus_tokens:
        print(f"{token.text:>8} | {token.pos_:>4} | {token.tag_:>3} | {token.lemma_}")

 runners | NOUN | NNS | runner
 running | VERB | VBG | run
     ran | VERB | VBD | run
    cars | NOUN | NNS | car
  better |  ADJ | JJR | well


## 한국어 형태소 분석과 기본형 정규화

한국어 용언은 어간과 어미가 결합해 다양한 활용형을 만든다. `기다리다`의 문법적 어간은 `기다리-`이고 어미는 `-다`이지만 실제 문장에서는 `기다리고`, `기다렸다`처럼 어미와 선어말어미가 결합한다. `긋다 → 그어`처럼 어간 모양도 달라질 수 있어 영어 PorterStemmer처럼 문자열 끝만 자르는 방식으로는 안정적인 정규화가 어렵다.

KoNLPy는 Okt, Komoran, Mecab 같은 한국어 형태소 분석기를 파이썬에서 사용할 수 있게 한다. 분석기마다 사전, 품사 체계와 분해 규칙이 달라 같은 문장도 결과가 다를 수 있다.

- Okt의 `stem=True`는 용언을 `기다리다`처럼 **기본형에 가까운 형태**로 정규화한다. `기다리-`와 같은 문법적 어간만 반환하는 기능으로 해석하면 안 된다.
- Komoran과 Mecab의 `morphs()`는 **분석된 형태소 목록**을 반환한다. WordNet의 `lemmatize()`처럼 모든 단어를 표제어로 복원하는 API와는 역할이 다르다.
- 분석기 출력은 사전과 규칙에 따른 예측이므로 원문 의미와 다른 기본형, 잘못된 분절과 품사를 표본 검사해야 한다.


### Okt로 표면형과 기본형 비교

`Okt.pos(text)`는 형태소와 품사의 튜플 목록을, `Okt.nouns(text)`는 명사 목록을 반환한다. `Okt.morphs(text, stem=False)`는 분석된 표면형을 유지하고 `stem=True`는 동사와 형용사를 기본형에 가까운 형태로 정규화한다.

같은 문장에서 두 옵션의 결과를 비교해 `기다리고`, `기다렸다`가 `기다리다`로 모이는지 확인한다. 명사 `기다림`이 유지되는지와 `쉽지 → 쉬다`처럼 원문의 기본형 `쉽다`와 다른 과도한 정규화가 생기는지도 함께 살핀다.


In [32]:
from konlpy.tag import Okt

okt = Okt()
short_text = "아버지가 방에 들어가신다."


print("품사 태그:", okt.pos(short_text))
print("명사:", okt.nouns(short_text))
print("형태소:", okt.morphs(short_text))
print("기본형 정규화:", okt.morphs(short_text, stem=True))

comparison_text = (
    "나는 버스를 기다리고 있다. 기다림은 쉽지 않다. "
    "나는 어제도 버스를 기다렸다."
)

surface_morphs = okt.morphs(comparison_text, stem=False)
normalized_morphs = okt.morphs(comparison_text, stem=True)
print("\nstem=False:", surface_morphs)
print("stem=True :", normalized_morphs)

품사 태그: [('아버지', 'Noun'), ('가', 'Josa'), ('방', 'Noun'), ('에', 'Josa'), ('들어가신다', 'Verb'), ('.', 'Punctuation')]
명사: ['아버지', '방']
형태소: ['아버지', '가', '방', '에', '들어가신다', '.']
기본형 정규화: ['아버지', '가', '방', '에', '들어가다', '.']

stem=False: ['나', '는', '버스', '를', '기다리고', '있다', '.', '기다림', '은', '쉽지', '않다', '.', '나', '는', '어제', '도', '버스', '를', '기다렸다', '.']
stem=True : ['나', '는', '버스', '를', '기다리다', '있다', '.', '기다림', '은', '쉬다', '않다', '.', '나', '는', '어제', '도', '버스', '를', '기다리다', '.']


### Komoran으로 어간과 어미 분해 확인

`Komoran.pos(text)`는 형태소와 세분화된 품사의 튜플 목록을, `Komoran.morphs(text)`는 품사 코드를 뺀 형태소 목록을 반환한다. 높임과 종결 표현이 포함된 문장에서 동사 어간 `VV`, 선어말어미 `EP`, 종결어미 `EF`가 서로 다른 형태소로 분리되는지 확인한다.

`morphs()`의 출력에 사전형 어미 `-다`가 없더라도 표제어가 자동으로 복원되었다고 해석하면 안 된다. Komoran의 이 기능은 한국어 형태소 구조를 분해하는 것이며 WordNet lemmatization과 출력 목적이 다르다.


In [33]:
from konlpy.tag import Komoran

# Komoran은 자체 사전과 품사 체계를 사용하므로 Okt와 같은 분절을 반환한다고 가정하지 않는다.
komoran = Komoran()
text = "아버지가 방에 들어가신다."

pos_tags = komoran.pos(text)

morphs = komoran.morphs(text)
print("품사 태그:", pos_tags)
print("형태소:", morphs)

품사 태그: [('아버지', 'NNG'), ('가', 'JKS'), ('방', 'NNG'), ('에', 'JKB'), ('들어가', 'VV'), ('시', 'EP'), ('ㄴ다', 'EF'), ('.', 'SF')]
형태소: ['아버지', '가', '방', '에', '들어가', '시', 'ㄴ다', '.']


### MeCab 품사와 사전 정보 확인

KoNLPy의 `Mecab`은 MeCab 형태소 분석기를 파이썬에서 사용하는 클래스이다. 이 클래스가 한국어를 분석하려면 Python 바인딩인 `mecab-python3`와 한국어 사전인 `mecab-ko-dic`이 함께 필요하다.

`mecab_ko_dic.mecabrc`는 MeCab 설정 파일의 위치이고 `mecab_ko_dic.DICDIR`은 한국어 사전 폴더의 위치이다. 설정 파일을 `MECABRC` 환경 변수에 지정한 뒤 `Mecab(dicpath=...)`에 사전 폴더를 전달하면 KoNLPy가 pip로 설치한 한국어 사전을 사용한다.

`pos`, `morphs`, `nouns`는 각각 품사 튜플, 전체 형태소와 명사 목록을 반환한다. 결과에서는 일반 명사 `NNG`, 주격 조사 `JKS`, 동사 `VV`, 선어말어미 `EP`, 종결어미 `EF`가 어떻게 구분되는지 확인한다. 형태소가 분리되어도 동사가 항상 `-다`형 표제어로 복원되는 것은 아니므로 WordNet lemmatization과 같은 결과로 해석하면 안 된다.


In [1]:
import os

import mecab_ko_dic
from konlpy.tag import Mecab

os.environ["MECABRC"] = mecab_ko_dic.mecabrc

# DICDIR은 mecab-ko-dic 패키지가 설치한 한국어 사전 폴더
# dicpath에 이 경로를 전달하면 시스템 전역 사전 대신 현재 Python 환경의 사전을 사용한다.
# mecab = Mecab(dicpath=mecab_ko_dic.DICDIR)
dic_path = mecab_ko_dic.DICDIR.replace("\\", "/")

mecab = Mecab(dicpath=dic_path)
text = "아버지가 방에 들어갔습니다."

# pos()는 (형태소, 품사 코드) 튜플 목록을 반환
print("품사 태그:", mecab.pos(text))

# morphs()는 품사 코드를 제외한 전체 형태소 목록을 반환
print("형태소:", mecab.morphs(text))

# nouns()는 명사로 분석된 형태소만 반환
print("명사:", mecab.nouns(text))

# 현재 문장에 등장하는 일반 명사, 주격 조사, 동사와 어미의 품사 설명만 선택
selected_tag_names = ["NNG", "JKS", "VV", "EP", "EF"]

# tagset은 품사 코드를 한국어 설명에 연결한 사전
selected_tagset = {tag: mecab.tagset[tag] for tag in selected_tag_names}
print("주요 품사표:", selected_tagset)

# dicpath는 Mecab 인스턴스가 실제로 참조하는 mecab_ko_dic 패키지의 사전 폴더
print("사전 경로:", mecab.dicpath)

품사 태그: [('아버지', 'NNG'), ('가', 'JKS'), ('방', 'NNG'), ('에', 'JKB'), ('들어갔', 'VV+EP'), ('습니다', 'EF'), ('.', 'SF')]
형태소: ['아버지', '가', '방', '에', '들어갔', '습니다', '.']
명사: ['아버지', '방']
주요 품사표: {'NNG': '일반 명사', 'JKS': '주격 조사', 'VV': '동사', 'EP': '선어말어미', 'EF': '종결 어미'}
사전 경로: C:/Users/playdata2/miniforge3/envs/nlp_env/Lib/site-packages/mecab_ko_dic/dicdir


## 정리와 적용 판단

1. **속도와 검색 재현율이 우선이면 stemming을 후보로 둔다.** `swiftli` 같은 비단어와 `org` 같은 과도한 축약을 표본 검사한다.
2. **문법적 기본형과 해석 가능성이 중요하면 lemmatization을 후보로 둔다.** WordNet은 품사를 명시적으로 전달하고 spaCy는 문맥에서 품사를 예측하므로 두 도구의 결과가 다를 수 있다.
3. **한국어는 형태소 분석기의 반환값을 먼저 확인한다.** Okt의 `stem=True`는 기본형 정규화에 가깝고, Komoran과 Mecab의 `morphs()`는 형태소 분해 결과이다.
4. **정규화 전후를 실험으로 결정한다.** 어휘 수, 대표 토큰, OOV 비율과 검증 성능을 비교하고 의미가 달라진 사례가 있으면 규칙·사전·분석기를 조정한다.

전처리 규칙과 분석기 선택은 Train·Validation 데이터에서 결정한다. 최종 Test 데이터의 결과에 맞춰 정규화 규칙을 반복 수정하면 평가 정보가 전처리 결정에 유입될 수 있으므로 Test 데이터는 마지막 평가에만 사용한다.
